In [2]:
import requests
import pandas as pd
import os 
from dotenv import load_dotenv  

In [3]:
found = load_dotenv()
print(found)

api_key = os.getenv("FRED_API_KEY")

if api_key is None:
    raise ValueError("FRED_API_KEY not found.")

True


In [4]:
url = "https://api.stlouisfed.org/fred/series/observations"
params = {"series_id":"CPIAUCSL", "api_key": api_key, "file_type":"json", "observation_start":"2005-01-01"}

response = requests.get(url, params=params, timeout=10)
response.raise_for_status()

data = response.json()
print(type(data), data.keys())

for obs in data["observations"]:
    print(obs["date"], obs["value"])

<class 'dict'> dict_keys(['realtime_start', 'realtime_end', 'observation_start', 'observation_end', 'units', 'output_type', 'file_type', 'order_by', 'sort_order', 'count', 'offset', 'limit', 'observations'])
2005-01-01 191.600
2005-02-01 192.400
2005-03-01 193.100
2005-04-01 193.700
2005-05-01 193.600
2005-06-01 193.700
2005-07-01 194.900
2005-08-01 196.100
2005-09-01 198.800
2005-10-01 199.100
2005-11-01 198.100
2005-12-01 198.100
2006-01-01 199.300
2006-02-01 199.400
2006-03-01 199.700
2006-04-01 200.700
2006-05-01 201.300
2006-06-01 201.800
2006-07-01 202.900
2006-08-01 203.800
2006-09-01 202.800
2006-10-01 201.900
2006-11-01 202.000
2006-12-01 203.100
2007-01-01 203.437
2007-02-01 204.226
2007-03-01 205.288
2007-04-01 205.904
2007-05-01 206.755
2007-06-01 207.234
2007-07-01 207.603
2007-08-01 207.667
2007-09-01 208.547
2007-10-01 209.190
2007-11-01 210.834
2007-12-01 211.445
2008-01-01 212.174
2008-02-01 212.687
2008-03-01 213.448
2008-04-01 213.942
2008-05-01 215.208
2008-06-01 21

In [5]:
print(type(obs["value"]))
print(type(obs["date"]))

<class 'str'>
<class 'str'>


In [6]:
data["observations"][0]

{'realtime_start': '2026-10-22',
 'realtime_end': '2026-10-22',
 'date': '2005-01-01',
 'value': '191.600'}

In [7]:
df = pd.DataFrame(data["observations"])[["realtime_start","realtime_end","date", "value"]]
print(df.shape)
print(df.dtypes)
print(df.tail())

(261, 4)
realtime_start    str
realtime_end      str
date              str
value             str
dtype: object
    realtime_start realtime_end        date    value
256     2026-10-22   2026-10-22  2026-05-01  333.979
257     2026-10-22   2026-10-22  2026-06-01  332.568
258     2026-10-22   2026-10-22  2026-07-01  332.813
259     2026-10-22   2026-10-22  2026-08-01  334.131
260     2026-10-22   2026-10-22  2026-09-01  336.298


In [8]:
df_nulos = df[df["value"] == "."]

nulos = (df["value"]==".").sum()
print(nulos)
print(df_nulos)

1
    realtime_start realtime_end        date value
249     2026-10-22   2026-10-22  2025-10-01     .


In [9]:
FRED_BASE_URL = "https://api.stlouisfed.org/fred/series/observations"

def fetch_fred_series(series_id, start_date, api_key):
    """Fetch observations for a FRED series and return the raw JSON response."""

    params = {"series_id": series_id,
               "api_key": api_key, 
               "file_type": "json", 
               "observation_start": start_date
               }

    response = requests.get(url, params=params, timeout=10)
    response.raise_for_status()

    # Validation of content type
    content_type = response.headers.get("Content-Type","")

    if "application/json" not in content_type:
        raise ValueError(
            f"Expected JSON but received '{content_type}' "
            f"(status {response.status_code}). "
            f"Response starts with: {response.text[:200]}"
        )
    
    return response.json()
     

In [10]:
#Test with wrong "series_id"
fetch_fred_series("CPIAUCSL", "2005-01-01", api_key)

{'realtime_start': '2026-10-22',
 'realtime_end': '2026-10-22',
 'observation_start': '2005-01-01',
 'observation_end': '9999-12-31',
 'units': 'lin',
 'output_type': 1,
 'file_type': 'json',
 'order_by': 'observation_date',
 'sort_order': 'asc',
 'count': 261,
 'offset': 0,
 'limit': 100000,
 'observations': [{'realtime_start': '2026-10-22',
   'realtime_end': '2026-10-22',
   'date': '2005-01-01',
   'value': '191.600'},
  {'realtime_start': '2026-10-22',
   'realtime_end': '2026-10-22',
   'date': '2005-02-01',
   'value': '192.400'},
  {'realtime_start': '2026-10-22',
   'realtime_end': '2026-10-22',
   'date': '2005-03-01',
   'value': '193.100'},
  {'realtime_start': '2026-10-22',
   'realtime_end': '2026-10-22',
   'date': '2005-04-01',
   'value': '193.700'},
  {'realtime_start': '2026-10-22',
   'realtime_end': '2026-10-22',
   'date': '2005-05-01',
   'value': '193.600'},
  {'realtime_start': '2026-10-22',
   'realtime_end': '2026-10-22',
   'date': '2005-06-01',
   'value': 

In [11]:
import yfinance as yf 

spy_df_ajust = yf.download("SPY", start="2006-01-01", auto_adjust=True)
print(spy_df_ajust.tail())

[*********************100%***********************]  1 of 1 completed

Price            Close        High         Low        Open    Volume
Ticker             SPY         SPY         SPY         SPY       SPY
Date                                                                
2026-10-19  766.479980  770.460022  765.820007  767.229980  38680100
2026-10-20  769.049988  772.289978  764.250000  765.859985  35048400
2026-10-21  773.090027  774.440002  768.400024  771.169983  31943800
2026-10-22  763.960022  767.710022  761.169983  766.369995  54117100
2026-10-23  759.969971  763.469971  758.679993  759.900024  33377700


In [25]:

spy_df_noajust = yf.download("SPY", start="2006-01-01",multi_level_index=False, auto_adjust=False)
spy_df_noajust.head()

[*********************100%***********************]  1 of 1 completed


,Adj Close,Close,High,Low,Open,Volume
Date,,,,,,
2006-01-03,86.785408,126.699997,127.000000,124.389999,125.190002,73256700
2006-01-04,87.196396,127.300003,127.489998,126.699997,126.860001,51899600
2006-01-05,87.251190,127.379997,127.589996,126.879997,127.150002,47307500
2006-01-06,87.977257,128.440002,128.580002,127.360001,128.020004,62885900
2006-01-09,88.203316,128.770004,129.059998,128.380005,128.419998,43527400


In [13]:
adj_profit = spy_df_ajust.iloc[-1]["Close"] / spy_df_ajust.iloc[0]["Close"] - 1
non_adj_profit = spy_df_noajust.iloc[-1]["Close"] / spy_df_noajust.iloc[0]["Close"] - 1

print(f"Adjusted return: {adj_profit.item():.1%}")   
print(f"Non adjusted return: {non_adj_profit.item():.1%}")   

Adjusted return: 775.7%
Non adjusted return: 499.8%


In [23]:
tickers = "SPY, TLT, TIP, GLD, DBC, VNQ"
df = yf.download(tickers, start="2006-01-01", multi_level_index=False,auto_adjust=False)
df.head()

[*********************100%***********************]  6 of 6 completed


Price      Adj Close                                                         \
Ticker           DBC        GLD        SPY        TIP        TLT        VNQ   
Date                                                                          
2006-01-03       NaN  53.119999  86.785416  54.848961  47.121235  25.602392   
2006-01-04       NaN  53.299999  87.196426  54.854286  47.234219  25.792011   
2006-01-05       NaN  52.340000  87.251183  54.694611  47.177731  25.990040   
2006-01-06       NaN  53.720001  87.977242  54.667965  47.090473  26.234449   
2006-01-09       NaN  54.599998  88.203300  54.731869  47.121235  26.457771   

Price      Close                                     ...        Open  \
Ticker       DBC        GLD         SPY         TIP  ...         SPY   
Date                                                 ...               
2006-01-03   NaN  53.119999  126.699997  103.040001  ...  125.190002   
2006-01-04   NaN  53.299999  127.300003  103.050003  ...  126.860001   
2006-01-05   NaN  52.340000  127.379997  102.750000  ...  127.150002   
2006-01-06   NaN  53.720001  128.440002  102.699997  ...  128.020004   
2006-01-09   NaN  54.599998  128.770004  102.820000  ...  128.419998   

Price                                        Volume                     \
Ticker             TIP        TLT        VNQ    DBC      GLD       SPY   
Date                                                                     
2006-01-03  102.690002  91.589996  59.939999    NaN  4606500  73256700   
2006-01-04  103.010002  91.540001  61.209999    NaN  5287900  51899600   
2006-01-05  102.769997  91.879997  61.240002    NaN  7661800  47307500   
2006-01-06  102.720001  91.769997  62.009998    NaN  4629800  62885900   
2006-01-09  102.639999  91.699997  62.599998    NaN  5394200  43527400   

Price                               
Ticker         TIP      TLT    VNQ  
Date                                
2006-01-03  257000  1392700  70300  
2006-01-04  162400   387600  75000  
2006-01-05  169200   453400  31200  
2006-01-06  315500  1193700  36000  
2006-01-09  288300   413600  49200  

[5 rows x 36 columns]

In [15]:
df.shape

(5235, 36)

In [16]:
df.isna().sum()

Price      Ticker
Adj Close  DBC       23
           GLD        0
           SPY        0
           TIP        0
           TLT        0
           VNQ        0
Close      DBC       23
           GLD        0
           SPY        0
           TIP        0
           TLT        0
           VNQ        0
High       DBC       23
           GLD        0
           SPY        0
           TIP        0
           TLT        0
           VNQ        0
Low        DBC       23
           GLD        0
           SPY        0
           TIP        0
           TLT        0
           VNQ        0
Open       DBC       23
           GLD        0
           SPY        0
           TIP        0
           TLT        0
           VNQ        0
Volume     DBC       23
           GLD        0
           SPY        0
           TIP        0
           TLT        0
           VNQ        0
dtype: int64

In [20]:
df['Adj Close']['DBC'].first_valid_index()

Timestamp('2006-02-06 00:00:00')